# Flight Test Data Access Demo

This notebook demonstrates how to access the processed flight data (H5 files) generated by the Flight Test Dashboard backend.

In [ ]:
import pandas as pd
import requests
import matplotlib.pyplot as plt
import os

# Backend API URL
API_URL = "http://localhost:8000"

## 1. Fetch Events and Identify Processed Files
First, we retrieve the list of flight events from the API and look for files that have been successfully processed.

In [ ]:
# Fetch all events
response = requests.get(f"{API_URL}/events/")
events = response.json()

print(f"Found {len(events)} events.")

# Find a file with a processed H5 path
target_file_path = None
target_event_name = None

for event in events:
    if event["files"]:
        for file in event["files"]:
            if file["processed_file_path"]:
                target_file_path = file["processed_file_path"]
                target_event_name = event["name"]
                print(f"Found processed file for event '{target_event_name}': {target_file_path}")
                break
    if target_file_path:
        break
        
if not target_file_path:
    print("No processed files found. Please use the dashboard to add a file first.")

## 2. Load Data into Pandas
Using the path retrieved from the API, we can load the data directly into a Pandas DataFrame using `read_hdf`.

In [ ]:
if target_file_path and os.path.exists(target_file_path):
    # Load the H5 file
    # Note: 'flight_data' is the key used in the backend processor
    df = pd.read_hdf(target_file_path, key='flight_data')
    
    print("Data loaded successfully!")
    print(df.info())
    display(df.head())
else:
    print("File not found locally.")

## 3. Visualize Data
Now we can use standard Python plotting libraries to analyze the flight data.

In [ ]:
if 'df' in locals():
    plt.figure(figsize=(12, 6))
    
    plt.subplot(2, 1, 1)
    plt.plot(df['timestamp'], df['altitude'], label='Altitude')
    plt.title(f"Flight Data Analysis: {target_event_name}")
    plt.ylabel("Altitude (ft)")
    plt.legend()
    
    plt.subplot(2, 1, 2)
    plt.plot(df['timestamp'], df['airspeed'], label='Airspeed', color='orange')
    plt.ylabel("Airspeed (knots)")
    plt.xlabel("Time")
    plt.legend()
    
    plt.tight_layout()
    plt.show()